## Definição


### Contexto
Este dataset reúne registros relacionados a câncer colorretal, contendo variáveis demográficas, clínicas, comportamentais e de contexto de saúde por exemplo: estágio do câncer, tamanho do tumor, fatores de risco e acesso ao sistema de saúde.

### Problema / Objetivo
O objetivo é realizar uma análise exploratória para identificar padrões e possíveis associações entre atributos do dataset e indicadores de desfecho por exemplo `Survival_Prediction`, `Survival_5_years` e `Mortality`.

### Hipóteses (perguntas que a análise busca responder)
- A proporção de `Survival_Prediction` varia entre estágios (`Cancer_Stage`).   
- A proporção de `Survival_Prediction` varia entre `Early_Detection = 0/1`.   
- Variáveis binárias ( diabetes, tabagismo etc.) apresentam (ou não) diferenças relevantes no desfecho quando comparamos `x=0` vs `x=1`.


### Restrições/condições
O dataset é público e não contém informações pessoais diretas além de um identificador (`Patient_ID`), que será removido por não agregar valor analítico.


### Dicionário de atributos (principais)
- `Patient_ID`: identificador único do registro.
- `Age`: idade (anos).
- `Gender`: sexo (M/F).
- `Cancer_Stage`: estágio do câncer (Localized/Regional/Metastatic).
- `Tumor_Size_mm`: tamanho do tumor em milímetros.
- `Family_History`: histórico familiar (Yes/No).
- `Smoking_History`: tabagismo (Yes/No).
- `Alcohol_Consumption`: consumo de álcool (Yes/No).
- `Diabetes`: diabetes (Yes/No).
- `Inflammatory_Bowel_Disease`: doença inflamatória intestinal (Yes/No).
- `Genetic_Mutation`: mutação genética (Yes/No).
- `Early_Detection`: detecção precoce (Yes/No).
- `Treatment_Type`: tipo de tratamento (categorias).
- `Healthcare_Costs`: custos de saúde (numérico).
- `Healthcare_Access`: acesso à saúde (Low/Moderate/High).
- `Insurance_Status`: segurado ou não (categorias).
- `Survival_Prediction`: indicador de desfecho analisado (Yes/No ou 0/1).
``

## Configuração do ambiente e carregamento do dataset

**Objetivo:** preparar o ambiente de trabalho e garantir que o dataset foi carregado corretamente antes de iniciar qualquer análise exploratória ou pré-processamento.

**O que foi feito neste bloco:**
- Importação das bibliotecas principais para manipulação e análise de dados (`pandas`, `numpy`).   
- Importação das bibliotecas de visualização (`matplotlib`, `seaborn`) e configuração do tema visual (`sns.set_theme()`), para padronizar o estilo dos gráficos.   
- Importação de componentes do `scikit-learn` (`OneHotEncoder`, `StandardScaler`, `ColumnTransformer`) que serão utilizados posteriormente em etapas de pré-processamento (codificação de variáveis categóricas e padronização de variáveis numéricas).   
- Leitura do arquivo `colorectal_cancer_dataset.csv` em um DataFrame chamado `dados`, seguida de uma prévia das três primeiras linhas para validação do carregamento e inspeção inicial do formato dos dados.   



In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

sns.set_theme()

caminho_arquivo = "colorectal_cancer_dataset.csv"
dados = pd.read_csv(caminho_arquivo)

print("Arquivo carregado com sucesso!")
display(dados.head(3))



## Análise inicial (estrutura do dataset)

**O que foi feito:**  
- Identificação do tamanho do dataset (linhas/colunas).  
- Listagem dos atributos disponíveis.  
- Verificação dos tipos (`object`, `int`, `float`).  
- Visualização das primeiras linhas.

**Por quê:**  
- Ajuda a entender o “formato” do dado: quais atributos são numéricos e quais são categóricos.  
- Ajuda a antecipar necessidades de pré-processamento ( encoding para texto, escalonamento para números).  


In [ ]:
quantidade_linhas, quantidade_colunas = dados.shape
print("Quantidade de linhas (registros):", quantidade_linhas)
print("Quantidade de colunas (atributos):", quantidade_colunas)

lista_colunas = dados.columns.tolist()
print("Lista de colunas:")
for indice, nome_coluna in enumerate(lista_colunas, start=1):
    print(f"{indice:02d} - {nome_coluna}")

tipos = dados.dtypes
print("\nTipos por coluna:")
display(tipos)

relatorio_tipos = (
    pd.DataFrame({"coluna": dados.columns, "tipo": dados.dtypes.astype(str)})
    .sort_values("tipo")
)
display(relatorio_tipos)

display(dados.head(10))


## Valores faltantes (missing)

**O que foi feito:** contagem de valores faltantes por coluna e no total (em número e porcentagem).  
**Por quê:** valores faltantes podem exigir tratamento (remoção, imputação, categoria “desconhecido” etc.).  

**Regra (como seria tratado se existisse):**  
- Numéricos → imputação por mediana/média  
- Categóricos → imputação por moda ou “Unknown”  
- Se faltantes forem muito altos em uma coluna → considerar remover a coluna

**Observação:** O total de faltantes foi 0, então não foi necessário imputar/remover por missing.



In [ ]:
faltantes_por_coluna = dados.isna().sum().sort_values(ascending=False)
print("Top 10 colunas com mais faltantes:")
display(faltantes_por_coluna.head(10))

faltantes_pct = (dados.isna().sum() / len(dados) * 100).round(2)
relatorio_faltantes = (
    pd.DataFrame({"faltantes": dados.isna().sum(), "faltantes_%": faltantes_pct})
    .sort_values("faltantes", ascending=False)
)

print("Relatório de faltantes (top 15):")
display(relatorio_faltantes.head(15))

total_faltantes = int(dados.isna().sum().sum())
print("Total de valores faltantes no dataset inteiro:", total_faltantes)


## Duplicatas (integridade)

**O que foi feito:**  
- Verificação de linhas duplicadas (dataset inteiro).  
- Verificação de duplicidade do identificador (`Patient_ID`) apenas se a coluna existir.

**Por quê:** duplicatas podem distorcer estatísticas e gráficos, além de induzir conclusões incorretas.

**Nota importante:** como o `Patient_ID` foi removido definitivamente para análise (por ser identificador), checagens futuras nessa coluna não se aplicam.

In [ ]:
duplicadas_linhas = int(dados.duplicated().sum())
print("Quantidade de linhas duplicadas:", duplicadas_linhas)

if "Patient_ID" in dados.columns:
    duplicados_patient_id = int(dados["Patient_ID"].duplicated().sum())
    print("Quantidade de Patient_ID duplicados:", duplicados_patient_id)
else:
    print("Patient_ID não existe mais no dataset (foi removido).")

## Separação de atributos por tipo

**O que foi feito:** separação automática de atributos numéricos e categóricos.  
**Por quê:** isso guia o pré-processamento:
- Numéricos: podem exigir normalização/padronização/transformação log.  
- Categóricos: podem exigir encoding (one-hot/ordinal).  
- Binários Yes/No: podem virar 0/1.


In [ ]:
colunas_numericas = dados.select_dtypes(include=[np.number]).columns.tolist()
colunas_categoricas = dados.select_dtypes(exclude=[np.number]).columns.tolist()

print("Colunas numéricas:", colunas_numericas)
print("Quantidade de colunas numéricas:", len(colunas_numericas))

print("\nColunas categóricas:", colunas_categoricas)
print("Quantidade de colunas categóricas:", len(colunas_categoricas))


## Estatísticas descritivas (atributos numéricos)

**O que foi feito:** cálculo de mínimo, máximo, média, mediana, moda, desvio padrão e quantidade de faltantes para colunas numéricas.  
**Por quê:** permite entender escala, dispersão e possíveis valores discrepantes/outliers.  


In [ ]:
def resumo_numerico(dados, colunas):
    lista_saida = []
    for coluna in colunas:
        serie = dados[coluna]
        lista_saida.append({
            "coluna": coluna,
            "min": serie.min(),
            "max": serie.max(),
            "media": serie.mean(),
            "mediana": serie.median(),
            "moda": serie.mode().iloc[0] if not serie.mode().empty else np.nan,
            "desvio_padrao": serie.std(ddof=1),
            "faltantes": int(serie.isna().sum())
        })
    return pd.DataFrame(lista_saida).set_index("coluna")

relatorio_numericos = resumo_numerico(dados, colunas_numericas)
display(relatorio_numericos)

## Checagem de consistência (min/max)

**O que foi feito:** inspeção rápida de valores mínimos e máximos para detectar valores fora do domínio esperado ( idade negativa, valores absurdos).  
**Por quê:** é uma forma simples e eficiente de identificar discrepâncias antes de qualquer transformação.
``

In [ ]:
for coluna in colunas_numericas:
    minimo = dados[coluna].min()
    maximo = dados[coluna].max()
    print(f"{coluna}: min={minimo} | max={maximo}")

## Remoção do identificador

**O que foi feito:** remoção definitiva do `Patient_ID`.  
**Por quê:** identificador não representa característica clínica/comportamental e pode atrapalhar análises (não agrega valor explicativo).  

Após remover colunas, recalculei `colunas_numericas` e `colunas_categoricas` para evitar erros em gráficos/análises posteriores.


In [ ]:
dados = dados.drop(columns=["Patient_ID"], errors="ignore")

colunas_numericas = dados.select_dtypes(include=[np.number]).columns.tolist()
colunas_categoricas = dados.select_dtypes(exclude=[np.number]).columns.tolist()

print("Dataset após remoção do ID:", dados.shape)
print("Patient_ID existe?", "Patient_ID" in dados.columns)

## Limpeza de texto (padronização)

**O que foi feito:** remoção de espaços extras nas colunas de texto (`strip`).  
**Por quê:** evita categorias duplicadas por erro de formatação ( "Yes" vs "Yes ").  
**Impacto:** melhora a qualidade de contagens, agrupamentos e visualizações por categoria.
``

In [ ]:
colunas_texto = dados.select_dtypes(include=["object"]).columns.tolist()

for coluna in colunas_texto:
    dados[coluna] = dados[coluna].astype("string").str.strip()


## Conversão de variáveis binárias (Yes/No → 0/1)

**O que foi feito:**  
- Identificação automática das colunas que possuem apenas `Yes/No`  
- Conversão para `0/1` com limpeza (`strip`, padronização de caixa)  
- Tratamento de valores não convertidos (se houver) preenchendo com a moda antes de converter para inteiro.

**Por quê:**  
- Modelos e análises numéricas trabalham melhor com 0/1.  
- Reduz ambiguidade em comparação com texto.  
- Evita falhas de conversão por pequenos problemas de padronização.



In [ ]:
colunas_binarias = []

for coluna in dados.columns:
    if str(dados[coluna].dtype) in ["object", "string"]:
        valores = set(dados[coluna].dropna().unique())
        if valores.issubset({"Yes", "No"}):
            colunas_binarias.append(coluna)

print("Colunas binárias encontradas:", colunas_binarias)
print("Quantidade:", len(colunas_binarias))


def converter_yes_no_para_0_1(serie):
    texto = serie.astype("string").str.strip().str.lower()
    convertido = texto.map({"no": 0, "yes": 1})
    return convertido


for coluna in colunas_binarias:
    dados[coluna] = converter_yes_no_para_0_1(dados[coluna])

    if dados[coluna].isna().any():
        moda = dados[coluna].mode(dropna=True)[0]
        dados[coluna] = dados[coluna].fillna(moda)

    dados[coluna] = dados[coluna].astype("int8")

print("Conversão concluída!")
display(dados[colunas_binarias].head(5))


## Encoding ordinal (Low/Moderate/High)

**O que foi feito:** mapeamento de categorias ordinais para números (Low=0, Moderate=1, High=2).  
**Por quê:** essas categorias têm ordem natural, então faz sentido representar com valores ordenados (diferente de one-hot, que não preserva ordem).  
**Visão criada:** `dados_ordinal` (dataset com ordinais convertidas).


In [ ]:
mapa_ordinal = {
    "Low": 0,
    "Moderate": 1,
    "High": 2
}

colunas_ordinais = ["Diet_Risk", "Physical_Activity", "Healthcare_Access"]
colunas_ordinais = [c for c in colunas_ordinais if c in dados.columns]

dados_ordinal = dados.copy()

for coluna in colunas_ordinais:
    dados_ordinal[coluna] = dados_ordinal[coluna].map(mapa_ordinal)

## One-hot encoding + padronização de numéricos

**O que foi feito:**  
- One-hot encoding nas variáveis categóricas nominais ( país, tratamento, estágio, etc.).  
- Padronização (z-score) das variáveis numéricas com `StandardScaler`.

**Por quê:**  
- One-hot transforma categorias em variáveis numéricas sem impor “ordem falsa”.  
- Padronização coloca numéricos na mesma escala (média 0, desvio 1), o que facilita análises futuras e evita que uma variável domine por escala.

**Visão criada:** `X_processado` (matriz pronta para análises futuras).


In [ ]:
coluna_alvo = "Survival_Prediction"

X = dados_ordinal.drop(columns=[coluna_alvo], errors="ignore")
y = dados_ordinal[coluna_alvo].copy()

colunas_numericas = X.select_dtypes(include=[np.number]).columns.tolist()
colunas_categoricas = X.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessador = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), colunas_numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), colunas_categoricas)
    ],
    remainder="drop"
)

X_processado = preprocessador.fit_transform(X)

print("Shape de X antes:", X.shape)
print("Shape de X após one-hot + scaler:", X_processado.shape)


## Transformação logarítmica

**O que foi feito:** criação de `Healthcare_Costs_log1p = log(1 + Healthcare_Costs)`.  
**Por quê:** custos costumam ter cauda longa (muitos valores médios e poucos muito altos). A transformação log reduz assimetria e pode melhorar interpretações e análises futuras.  
**Visão criada:** `dados_log` (dataset com coluna transformada).


In [ ]:
dados_log = dados_ordinal.copy()

if "Healthcare_Costs" in dados_log.columns:
    dados_log["Healthcare_Costs_log1p"] = np.log1p(dados_log["Healthcare_Costs"])



##Visualizações

### Gráfico 1 - Histograma das variáveis numéricas

**O que mostra:**  
A distribuição (forma) de variáveis numéricas selecionadas. Aqui aparecem tanto variáveis contínuas (`Age`, `Tumor_Size_mm`) quanto variáveis originalmente binárias que já foram convertidas para 0/1 (`Family_History`, `Smoking_History`, `Alcohol_Consumption`). Também aparece `Diet_Risk`, que é categórica/ordinal e foi plotada para visualizar a frequência por nível.

**Como interpretar:**  
- Distribuições muito concentradas em poucos valores sugerem variáveis discretas/binárias, onde faz mais sentido olhar **frequência** e **proporções** por grupo.  
- Para variáveis ordinais ( Low/Moderate/High), o interesse é comparar **qual nível é mais frequente**.

**O que foi observado neste dataset:**  
- `Age`: a massa principal está concentrada aproximadamente entre 50 e 90, com distribuição relativamente “espalhada” nessa faixa (sem um pico único muito dominante).  
- `Tumor_Size_mm`: a distribuição parece bem distribuída ao longo do intervalo (sem cauda longa evidente), sugerindo pouca assimetria visual.  
- `Family_History`, `Smoking_History` e `Alcohol_Consumption`: apresentam padrão típico de variável binária.
- `Diet_Risk`: o nível **Moderate** aparece como o mais frequente visualmente, com `Low` e `High` menores.


In [ ]:
figura, eixos = plt.subplots(2, 3, figsize=(16, 8))
eixos = eixos.ravel()

for eixo, coluna in zip(eixos, colunas_numericas[:6]):
    sns.histplot(dados[coluna], bins=30, kde=True, ax=eixo)
    eixo.set_title(f"Distribuição de {coluna}")
    eixo.set_xlabel(coluna)
    eixo.set_ylabel("Frequência")

plt.tight_layout()
plt.show()

### Gráfico 2 - Frequência de variáveis categóricas

**O que mostra:**  
A contagem de ocorrências por categoria para variáveis categóricas selecionadas (`Cancer_Stage`, `Treatment_Type`, `Obesity_BMI` e `Survival_Prediction`). Ajuda a entender quais categorias são mais comuns e se há desequilíbrios relevantes.

**Como interpretar:**  
- Barras maiores indicam categorias mais frequentes.  
- Categorias muito raras podem gerar ruído e dificultar análises comparativas.  
- Para a variável de interesse (`Survival_Prediction`), uma diferença grande entre as classes pode indicar desbalanceamento.

**O que foi observado neste dataset:**  
- Em `Cancer_Stage`, as categorias **Regional** e **Localized** aparecem com contagens muito próximas e maiores do que **Metastatic**, que é visivelmente menor.   
- Em `Treatment_Type`, **Surgery** é a categoria mais frequente; **Radiotherapy** aparece como a menos frequente.   
- Em `Obesity_BMI`, **Overweight** é a categoria mais frequente, enquanto **Normal** e **Obese** aparecem com contagens semelhantes entre si.   
- Em `Survival_Prediction`, a classe **1** aparece mais frequente do que a classe **0**, indicando um desbalanceamento moderado entre as classes.   


In [ ]:
colunas_para_grafico = ["Cancer_Stage", "Treatment_Type", "Obesity_BMI", "Survival_Prediction"]

figura, eixos = plt.subplots(2, 2, figsize=(14, 10))
eixos = eixos.ravel()

for eixo, coluna in zip(eixos, colunas_para_grafico):
    contagem = dados[coluna].value_counts()
    sns.barplot(x=contagem.index, y=contagem.values, ax=eixo)
    eixo.set_title(f"Frequência de {coluna}")
    eixo.set_xlabel(coluna)
    eixo.set_ylabel("Contagem")
    eixo.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


### Gráfico 3 - Top 10 países

**O que mostra:** os 10 países com mais registros no dataset.  
**Como interpretar:**  
- Ajuda a entender se o dataset é dominado por poucos países (possível viés de representatividade).  


In [ ]:
plt.figure(figsize=(10, 5))
top10 = dados["Country"].value_counts().head(10)
plt.bar(top10.index.astype(str), top10.values)
plt.title("Top 10 países (Country)")
plt.ylabel("Contagem")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


### Gráfico 4 - Taxa de Survival_Prediction = 1 por Cancer_Stage

**O que mostra:**  
A proporção (taxa) de `Survival_Prediction = 1` para cada categoria de `Cancer_Stage` (por exemplo: Localized, Metastatic, Regional).

**Como interpretar:**  
- Barras mais altas indicam maior proporção de `Survival_Prediction = 1` naquele estágio.  
- O eixo Y está com **zoom**, então diferenças pequenas ficam mais visíveis e comparáveis entre os estágios.

**O que foi observado neste dataset:**  
- As taxas entre os estágios aparecem **muito próximas**, com variações discretas.  
- Isso sugere que, analisando `Cancer_Stage` isoladamente, há **baixa variação** na proporção de `Survival_Prediction = 1` entre os grupos.


In [ ]:
col_estagio = "Cancer_Stage"
col_alvo = "Survival_Prediction"

prop_estagio = pd.crosstab(dados[col_estagio], dados[col_alvo], normalize="index")

print("Tabela de proporções (Cancer_Stage x Survival_Prediction):")
display(prop_estagio)

plt.figure(figsize=(7,4))
plt.bar(prop_estagio.index.astype(str), prop_estagio[1])
plt.title("Taxa de Survival_Prediction = 1 por Cancer_Stage")
plt.xlabel("Cancer_Stage")
plt.ylabel("Proporção de 1")
plt.ylim(prop_estagio[1].min() - 0.01, prop_estagio[1].max() + 0.01)  # zoom
plt.tight_layout()
plt.show()

### Gráfico 5 - Taxa de Survival_Prediction = 1 por Early_Detection

**O que mostra:**  
A proporção (taxa) de `Survival_Prediction = 1` para cada grupo de `Early_Detection` (0 e 1).

**Como interpretar:**  
- Se a barra de `Early_Detection = 1` for maior do que a de `Early_Detection = 0`, isso sugere uma associação positiva entre detecção precoce e o desfecho `Survival_Prediction = 1`.  
- O eixo Y está com **zoom**, então pequenas diferenças ficam mais visíveis.

**O que foi observado neste dataset:**  
- A taxa de `Survival_Prediction = 1` em `Early_Detection = 1` aparece **ligeiramente maior** do que em `Early_Detection = 0`.  
- Apesar disso, a diferença visual é **pequena** (as barras ficam muito próximas), indicando que `Early_Detection`, isoladamente, não altera muito a proporção do desfecho no conjunto analisado.


In [ ]:
col_detec = "Early_Detection"
col_alvo = "Survival_Prediction"

prop_detec = pd.crosstab(dados[col_detec], dados[col_alvo], normalize="index")


print("Tabela de proporções (Early_Detection x Survival_Prediction):")
display(prop_detec)


ordem_colunas = [c for c in [0, 1] if c in prop_detec.columns]
prop_detec = prop_detec[ordem_colunas]

plt.figure(figsize=(6,4))
plt.bar(prop_detec.index.astype(str), prop_detec[1])
plt.title("Taxa de Survival_Prediction = 1 por Early_Detection")
plt.xlabel("Early_Detection")
plt.ylabel("Proporção de 1")
plt.ylim(prop_detec[1].min() - 0.01, prop_detec[1].max() + 0.01)  # zoom
plt.tight_layout()
plt.show()


### Gráfico 6 - Diferença (p.p.) do desfecho por variáveis binárias

**O que mostra:** para cada variável binária, a diferença em pontos percentuais da taxa de `Survival_Prediction=1` quando a variável muda de 0 para 1.  
**Como interpretar:**  
- Barras positivas: maior taxa de desfecho=1 quando x=1.  
- Barras negativas: menor taxa de desfecho=1 quando x=1.  
- Barras próximas de zero: pouca diferença (baixa associação isolada).  

**O que foi observado neste dataset:**  
- As diferenças encontradas são **pequenas (todas abaixo de ~0,5 p.p.)**, indicando que essas variáveis binárias, analisadas isoladamente, têm **baixa variação** na proporção do desfecho.  
- No gráfico, `Diabetes` e `Survival_5_years` aparecem com as maiores diferenças positivas (ainda assim pequenas), enquanto `Inflammatory_Bowel_Disease` e `Smoking_History` aparecem com diferenças negativas.


In [ ]:
coluna_alvo = "Survival_Prediction"

if coluna_alvo in colunas_binarias:
    colunas_binarias.remove(coluna_alvo)

dif_pp = []

for col in colunas_binarias:
    taxa = dados.groupby(col)[coluna_alvo].mean()

    chaves_zero = [0, 0.0, "0", False, "No", "no"]
    chaves_um   = [1, 1.0, "1", True,  "Yes", "yes"]

    chave0 = next((k for k in chaves_zero if k in taxa.index), None)
    chave1 = next((k for k in chaves_um   if k in taxa.index), None)

    if chave0 is not None and chave1 is not None:
        dif_pp.append((col, (taxa[chave1] - taxa[chave0]) * 100))

dif_pp = pd.DataFrame(dif_pp, columns=["variavel", "diferença_pp"]).sort_values("diferença_pp")

print("Linhas em dif_pp:", len(dif_pp))
display(dif_pp.head(10))

plt.figure(figsize=(10, 5))
plt.bar(dif_pp["variavel"], dif_pp["diferença_pp"])
plt.axhline(0, color="black", linewidth=1)
plt.title("Diferença na taxa de Survival_Prediction (p.p.) entre x=1 e x=0")
plt.ylabel("Diferença (p.p.)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Gráfico 7 - Healthcare_Costs: distribuição original vs log1p

**O que mostra:** comparação direta da distribuição de `Healthcare_Costs` no formato original e após aplicar `log1p` (log(1 + x)).

**Como interpretar:**  
- Se o gráfico original apresentar cauda longa (muitos valores menores e poucos valores muito altos), a distribuição tende a ser **assimétrica**.  
- A transformação `log1p` costuma **reduzir assimetria** e comprimir valores altos, deixando a distribuição mais “equilibrada”.

**O que foi observado neste dataset:**  
- No formato original, `Healthcare_Costs` apresenta uma distribuição visualmente **bem regular**, sem uma cauda longa muito marcada.  
- Após `log1p`, a distribuição passa a ficar visualmente “puxada” para valores mais altos do log, indicando que **neste caso** a transformação não está “corrigindo” uma assimetria evidente, mas sim alterando a forma da distribuição.

- O log foi mantida como **visão alternativa** (`Healthcare_Costs_log1p`) para comparação e para possíveis análises futuras, pois comprime a escala e pode facilitar comparações entre grupos.  
- Como não foi observada grande assimetria no custo original, a transformação log é tratada como **opcional**.

In [ ]:
if "Healthcare_Costs_log1p" not in dados.columns:
    dados["Healthcare_Costs_log1p"] = np.log1p(dados["Healthcare_Costs"])

figura, eixos = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(dados["Healthcare_Costs"], bins=30, kde=True, ax=eixos[0])
eixos[0].set_title("Healthcare_Costs (original)")
eixos[0].set_xlabel("Healthcare_Costs")
eixos[0].set_ylabel("Frequência")

sns.histplot(dados["Healthcare_Costs_log1p"], bins=30, kde=True, ax=eixos[1])
eixos[1].set_title("Healthcare_Costs (log1p)")
eixos[1].set_xlabel("Healthcare_Costs_log1p")
eixos[1].set_ylabel("Frequência")

plt.tight_layout()
plt.show()

# Conclusões da Análise Exploratória e Pré-processamento - Dataset de Câncer Colorretal

## Visão geral do dataset
- O conjunto de dados possui **167.497 registros** e **28 atributos** (incluindo `Survival_Prediction`).   
- Não foram identificadas **linhas duplicadas** na checagem inicial e também não houve duplicidade de `Patient_ID` quando ele estava presente.   
- Na checagem de qualidade realizada inicialmente, o dataset **não apresentou valores faltantes**.   


---

## Definição do problema (o que este trabalho buscou responder)
Este trabalho teve foco em **análise exploratória e pré-processamento**, buscando compreender padrões e possíveis associações entre variáveis clínicas, comportamentais e de contexto de saúde com o desfecho **`Survival_Prediction`**. O objetivo preparar os dados e construir uma narrativa analítica clara a partir das evidências do dataset.

### Hipóteses/perguntas investigadas
- A proporção de `Survival_Prediction` varia entre estágios (`Cancer_Stage`).   
- A proporção de `Survival_Prediction` varia entre `Early_Detection = 0/1`.   
- Variáveis binárias ( diabetes, tabagismo etc.) apresentam (ou não) diferenças relevantes no desfecho quando comparamos `x=0` vs `x=1`.

---

## Principais achados

### 1) Distribuição do desfecho (classe)
- `Survival_Prediction` apresenta distribuição **moderadamente desbalanceada**, com **Yes ≈ 100.437** e **No ≈ 67.060** (aprox. 60/40).   
**Interpretação:** não é um desbalanceamento extremo, mas é relevante documentar porque pode influenciar análises comparativas e (no futuro) eventuais modelos.   

### 2) Variáveis categóricas (frequências)
Algumas distribuições importantes observadas:
- `Cancer_Stage`: **Regional (~66.981)**, **Localized (~66.799)**, **Metastatic (~33.717)**.   
- `Treatment_Type`: **Surgery (~66.934)**, **Chemotherapy (~50.443)**, **Combination (~33.276)**, **Radiotherapy (~16.844)**.   
- `Obesity_BMI`: **Overweight (~67.168)**, **Normal (~50.190)**, **Obese (~50.139)**.   
- Top 10 países em volume incluem **USA (~25.927)**, **China (~17.525)** e **Brasil (~10.399)** (entre outros).   

**Interpretação:** essas frequências ajudam a identificar categorias dominantes e guiam decisões de codificação ( necessidade de one-hot e possível agrupamento de categorias raras em análises futuras).

### 3) Variáveis numéricas (estatísticas e faixas)
- `Age`: faixa **30–89**, média ~**69,2**.   
- `Tumor_Size_mm`: faixa **5–79**, média ~**42,0**.   
- `Healthcare_Costs`: faixa **25.000–119.999**, média ~**72.452**.   
- `Incidence_Rate_per_100K`: faixa **10–59**, média ~**34,5**.   
- `Mortality_Rate_per_100K`: faixa **5–29**, média ~**17,0**.   

**Interpretação:** a análise de min/max e estatísticas descritivas valida consistência de domínio e sinaliza onde transformações podem ser úteis ( custos tendem a ter escala alta e podem se beneficiar de padronização e/ou transformação).

### 4) Variáveis binárias: diferenças em pontos percentuais (x=1 vs x=0)
Ao comparar `p(Survival_Prediction=1 | x=1)` contra `p(Survival_Prediction=1 | x=0)` para variáveis binárias, foram observadas **diferenças pequenas** (em geral abaixo de ~0,5 ponto percentual). Exemplos:
- `Diabetes`: diferença ~**+0,46 p.p.**
- `Early_Detection`: diferença ~**+0,23 p.p.**
- Outras binárias ( histórico familiar, mutação genética, tabagismo) apresentaram diferenças próximas de zero.

**Interpretação:** isoladamente, essas variáveis binárias parecem ter **baixa separação** do desfecho.

---

## Conclusão
A análise exploratória permitiu compreender a estrutura do dataset, validar sua qualidade (sem duplicatas e sem faltantes na checagem inicial), caracterizar distribuições de variáveis numéricas e categóricas, e avaliar associações iniciais entre `Survival_Prediction` e atributos como `Cancer_Stage`, `Early_Detection` e variáveis binárias.
